# Assignment — LangChain Fundamentals
## Landscape through Tools

**Domain for this assignment: GreenPlate — a restaurant reservation and food ordering
assistant.** This is a deliberately different scenario from CineBot, used throughout your
notebooks — the goal is to prove you understand the *concepts*, not that you can copy-paste
code you've already seen with new variable names.

**Structure:** Part A is conceptual (no coding), Part B is coding exercises, both arranged from
easier to harder. Part C is a single capstone challenge that combines everything. Attempt
sections in order — later questions build on ideas from earlier ones.

**Before you start:** make sure your environment is set up (API key loaded) and you can run a
basic `model.invoke()` successfully.

**A note on collaboration:** discussing concepts with classmates is encouraged. Copying code
without understanding it will be obvious the moment a follow-up question asks you to modify or
explain it.


In [1]:
%pip install -qU langchain langchain-openai langgraph pydantic langchain-core
%pip install -qU rich

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    api_key = os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    print(f"OpenRouter API key set in environment variable OPENROUTER_API_KEY: {api_key[0:4]}")

OpenRouter API key set in environment variable OPENROUTER_API_KEY: sk-o


---
# Part A — Conceptual Questions

## A1. Foundations (Easy)

1. In your own words, explain the sentence: *"An agent is a model calling tools in a loop until
   a task is complete. A harness is everything around that loop."* What specifically counts as
   part of the "harness"?

Answer: Harness includes everythng around a model which consists of prompts(system or human), skills, memory, 
        agents/subagents,tools which help make the agent more productive and realistic.

2. Name the four products in the Lang family (LangChain, LangGraph, LangSmith, Deep Agents) and
   state, in one sentence each, what job each one does. Which one is fundamentally different in
   *kind* from the other three, and why?

3. If you found a 2026-dated tutorial using `AgentExecutor` or `initialize_agent`, what would
   you conclude, and what should you use instead?

4. Why does a `.env` file exist? What specifically goes wrong if you skip it and hardcode an API
   key directly into a notebook cell?

Answer: .env file is strictly used to maintain the senstive keys or credentials related to models or databases.Harcoding it will expose the sensitive keys 
        and info which is will harm the whole deisgn of the agent or prone to attacks.

5. What is the difference between a plain text prompt, a message-object list, and a
   dictionary-based message list? Give one situation where each is the natural choice.

## A2. Models, Messages, and Templates (Medium)

6. An `AIMessage` carries more than `.content`. Name at least four other fields or attributes it
   can carry, and what each one is actually useful for.

7. Explain why streaming produces `AIMessageChunk` objects instead of plain text fragments, and
   what property of these chunks makes them genuinely different from a string split into pieces.

8. What's the difference between `.batch()` and `.batch_as_completed()`? Describe a real
   situation where you would specifically want the second one over the first.

9. A `ToolMessage` has both a `.content` field and an `.artifact` field. What's the difference,
   and why would a RAG-style tool specifically want to use `.artifact`?

10. You're writing a `ChatPromptTemplate` whose system message needs to include a literal JSON
    example like `{"status": "ok"}`. What will go wrong if you paste that in directly, and how
    do you fix it?

11. What does `MessagesPlaceholder` do, and why can't you achieve the same result with a normal
    string-based template variable?

## A3. Structured Output (Medium-Hard)

12. Explain, in your own words, why asking a model to "please respond in JSON" via plain prompt
    instructions is fundamentally less reliable than using `with_structured_output()`.

13. What is `model.profile`, and how is it actually used internally when you call
    `with_structured_output()` without specifying a strategy explicitly?

14. A teammate writes:
    `model.with_structured_output(BookingRequest, strategy=ProviderStrategy(BookingRequest))`
    and asks you to code review it. What's wrong with this line, and what should it be instead?

15. What's the difference between structured output at the **model** level
    (`with_structured_output`) and at the **agent** level (`response_format` on `create_agent`)?
    Why does almost everything from Part 7 onward use the agent-level version?

16. You have a schema `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`.
    Explain what happens internally when the model receives a message that's genuinely
    ambiguous between the two schemas.

17. A schema field is defined as `party_size: int = Field(ge=1, le=20)`, and a customer message
    says "table for 50 please." Walk through, step by step, what happens inside the agent loop
    from the moment the model first proposes `party_size=50` to the moment a valid final answer
    is produced.

## A4. Tools (Hard)

18. A tool's docstring is described as "the tool's entire pitch to the model," not documentation
    for humans. Defend or challenge this claim — is there ever a case where the docstring
    genuinely doesn't matter much?

19. Explain what `ToolRuntime` actually hides from the model, and how LangChain knows to hide it
    (i.e., what specifically triggers this behavior)?

20. Compare `runtime.state`, `runtime.context`, and `runtime.store`. For each one, state: (a) how
    long the data persists, and (b) one concrete example of information that belongs there.

21. A tool accidentally declares a parameter named `config`. What actually happens when the
    agent tries to call it, and why is this a genuinely easy mistake to make by accident?

22. Explain the difference between a tool returning a plain string versus returning a `Command`.
    Give an original example (not from any notebook you've seen) of a situation that specifically
    requires `Command`, and explain why a plain string return wouldn't work for that case.

23. Describe, precisely, why `wrap_model_call`-based tool gating (making a tool invisible) is a
    stronger guarantee than instructing the model in the system prompt not to use a tool.

24. What is a **headless tool**, and how is its execution model fundamentally different from
    every other tool pattern covered in this course? Name one realistic capability that could
    only be implemented this way.


In [20]:
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore


from pydantic import BaseModel, Field, ValidationError
from typing import Literal


In [4]:
# OpenRouter exposes an OpenAI-compatible API, so ChatOpenAI works with a custom base_url
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)

---
# Part B — Coding Exercises

All exercises use **GreenPlate**, a restaurant reservation and food ordering assistant. Each
question gives you a blank code cell to work in — write your solution there and run it to
confirm it works before moving on.

## B1. Easy

**B1.1 — A basic tool.** Write a `@tool`-decorated function called `check_table_availability`
that takes a `party_size: int` and a `time_slot: str`, and returns a string saying whether a
table is available (you can hardcode fake availability data, e.g. tables available for parties
of 2-6 at "7:00 PM" and "8:30 PM" only). Include a proper docstring. Print the tool's `.name`,
`.description`, and `.args` to confirm it's built correctly.


In [5]:
# Your solution for B1.1



@tool
def check_table_availability(party_size:int,time_slot:str) -> str:
    """
    Check the availability of a table for a given party size and time slot.
    
    Args:
        party_size (int): The number of people in the party.
        time_slot (str): The desired time slot for the reservation (e.g., "7:00 PM").
    
    Returns:
        str: A message indicating whether a table is available or not.
    """
    # Simulated logic for checking table availability
    if party_size <= 4 and time_slot in ["6:00 PM", "7:00 PM", "8:00 PM"]:
        return f"Table available for {party_size} people at {time_slot}."
    else:
        return f"No table available for {party_size} people at {time_slot}. Please choose a different time slot or reduce the party size."

print("Tool Name:", check_table_availability.name)

Tool Name: check_table_availability


**B1.2 — A reusable prompt template.** Build a `ChatPromptTemplate` that generates a short,
enthusiastic description of a dish, given `{dish_name}` and `{cuisine_type}` as variables. Run
it with at least two different dish/cuisine combinations and print both results.


In [6]:


# Your solution for B1.2
def reusable_prompt_template():
    return ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant. Write a short, enthusiastic dish description."),
            ("human", "Describe {dish_name} from {cuisine_type} cuisine in 2-3 sentences.")
        ]
    )



prompt = reusable_prompt_template()

# Run 1
messages1 = prompt.format_messages(dish_name="Pasta", cuisine_type="Italian")
response1 = model.invoke(messages1)
print("Pasta / Italian:\n", response1.content, "\n")

# Run 2
messages2 = prompt.format_messages(dish_name="Sushi", cuisine_type="Japanese")
response2 = model.invoke(messages2)
print("Sushi / Japanese:\n", response2.content)



Pasta / Italian:
 Indulge in the heart of Italian cuisine with our delightful pasta, crafted from the finest durum wheat and water, resulting in a perfectly al dente texture. Each dish is a canvas for vibrant sauces, from rich marinara bursting with fresh tomatoes to creamy Alfredo that envelops every bite in velvety goodness. Paired with fresh herbs and artisanal cheeses, our pasta promises an authentic taste of Italy that will transport you straight to the sun-soaked streets of Rome! 

Sushi / Japanese:
 Sushi is a delightful Japanese culinary masterpiece that artfully combines vinegared rice with fresh, high-quality seafood, vibrant vegetables, and sometimes even tropical fruits. Each bite is a harmonious explosion of flavors and textures, from the melt-in-your-mouth fish to the satisfying crunch of pickled vegetables, all beautifully presented. Whether you savor it as nigiri, maki, or sashimi, sushi is a captivating experience that celebrates the essence of Japanese tradition and c

**B1.3 — A schema with a constrained field.** Define a Pydantic `BaseModel` called
`FoodOrder` with fields: `customer_name` (str), `dish_name` (str), `quantity` (int, must be
between 1 and 10), and `spice_level` (a `Literal` restricted to `"mild"`, `"medium"`, or
`"hot"`). Use `with_structured_output()` to extract a `FoodOrder` from this message: *"Hi, I'm
Karan, 2 butter chicken please, medium spice."* Print the result.


In [7]:
# Your solution for B1.3
class FoodOrder(BaseModel):
    customer_name: str
    dish_name: str
    quantity: int = Field(gt=0, lt=11, description="Quantity must be between 1 and 10")
    spice_level: Literal["mild", "medium", "hot"] = Field(description="Spice level must be one of: 'mild', 'medium', 'hot'")
    
response = model.with_structured_output(FoodOrder).invoke([SystemMessage(content="You are a helpful assistant that takes food orders."),
              HumanMessage(content="Hi, I'm Karan, 2 butter chicken please, medium spice")])

print("Structured Output:\n", response, "\n")

Structured Output:
 customer_name='Karan' dish_name='butter chicken' quantity=2 spice_level='medium' 



## B2. Medium

**B2.1 — Two schemas, one agent.** A restaurant assistant needs to handle both new reservations
and cancellations. Define `NewReservation` (customer_name, party_size, time_slot) and
`CancelReservation` (customer_name, time_slot) as separate Pydantic models. Build a
`create_agent` with `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`,
and test it with one clearly-a-reservation message and one clearly-a-cancellation message. Use
`isinstance()` to print which schema was chosen each time.


In [10]:
# Your solution for B2.1
from typing import Union
class NewReservation(BaseModel):
    customer_name: str
    party_size: int = Field(gt=0, lt=21, description="Party size must be between 1 and 20")
    time_slot: str = Field(description="Time slot must be in the format 'HH:MM AM/PM'")
    
class CancelReservation(BaseModel):
    customer_name: str
    time_slot: str = Field(description="Time slot must be in the format 'HH:MM AM/PM'")
    
agent = create_agent(
    model,
    tools=[],
    system_prompt="You are GreenPlate, a restaurant assistant. Decide whether the user wants to make a reservation or cancel one.",
    response_format=ToolStrategy(Union[NewReservation, CancelReservation]),
)

tests = [
    ("reservation", "Please book a table for 4 people at 7:00 PM for Maya."),
    ("cancellation", "Cancel my reservation for Alex at 8:30 PM."),
]

for label, message in tests:
    result = agent.invoke({"messages": [HumanMessage(content=message)]})
    structured = result["structured_response"]
    print(f"{label}: {type(structured).__name__}")
    print(structured)
    print("is NewReservation:", isinstance(structured, NewReservation))
    print("is CancelReservation:", isinstance(structured, CancelReservation))
    print()

reservation: NewReservation
customer_name='Maya' party_size=4 time_slot='7:00 PM'
is NewReservation: True
is CancelReservation: False

cancellation: CancelReservation
customer_name='Alex' time_slot='8:30 PM'
is NewReservation: False
is CancelReservation: True



**B2.2 — A tool with `args_schema`.** Define a Pydantic input schema called `OrderInput` with
`dish_name` (str, with a description), `quantity` (int, `ge=1, le=10`, with a description), and
`delivery_or_pickup` (a `Literal["delivery", "pickup"]`, defaulting to `"pickup"`). Build a tool
called `place_order` using `args_schema=OrderInput`. Print `place_order.args` to confirm the
schema came through correctly, including the constraint and the default.


In [11]:
# Your solution for B2.2
class OrderInput(BaseModel):
    dish_name: str = Field(description="Name of the dish to order")
    quantity: int = Field(ge=0, le=10, description="Quantity must be between 1 and 10")
    delivery_or_pickup: Literal["delivery", "pickup"] = Field(description="Must be either 'delivery' or 'pickup'",
                                                              default="pickup")
    
@tool(args_schema=OrderInput)
def place_order(dish_name: str, quantity: int, delivery_or_pickup: str) -> str:
    """
    Place an order for a dish with the specified quantity and delivery/pickup option.
    
    Args:
        dish_name (str): The name of the dish to order.
        quantity (int): The quantity of the dish to order (must be between 1 and 10).
        delivery_or_pickup (str): Must be either 'delivery' or 'pickup'.
    
    Returns:
        str: A confirmation message for the placed order.
    """
    return f"Order placed: {quantity} x {dish_name} for {delivery_or_pickup}."

print("Tool Name:", place_order.name)
print("Place order arguments schema:", place_order.args)

Tool Name: place_order
Place order arguments schema: {'dish_name': {'description': 'Name of the dish to order', 'title': 'Dish Name', 'type': 'string'}, 'quantity': {'description': 'Quantity must be between 1 and 10', 'maximum': 10, 'minimum': 0, 'title': 'Quantity', 'type': 'integer'}, 'delivery_or_pickup': {'default': 'pickup', 'description': "Must be either 'delivery' or 'pickup'", 'enum': ['delivery', 'pickup'], 'title': 'Delivery Or Pickup', 'type': 'string'}}


**B2.3 — Deliberately trigger a validation failure and watch self-correction.** Using the
`FoodOrder` schema from B1.3, build a `create_agent` with
`response_format=ToolStrategy(FoodOrder)`. Send a message asking for 15 units of a dish (this
should violate your `quantity` constraint). Print the full `result["messages"]` trace and point
out, in a comment, exactly where the self-correction happens.


In [18]:
# Your solution for B2.3
agent = create_agent(
    model,
    tools = [],
    system_prompt="You are GreenPlate, a restaurant assistant. Take the user's order and provide a structured response.",
    response_format=ToolStrategy(FoodOrder)
)

result = agent.invoke({"messages": [HumanMessage(content="I would like to place an order for 15 units of Margherita pizzas for delivery.")]})
print(result["messages"])

from pprint import PrettyPrinter
pp = PrettyPrinter(indent=4)
pp.pprint(result["messages"])

[HumanMessage(content='I would like to place an order for 15 units of Margherita pizzas for delivery.', additional_kwargs={}, response_metadata={}, id='15cb2e4d-cc47-4e7c-8c9c-cfc47b49af88'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 143, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 4.245e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.245e-05, 'upstream_inference_prompt_cost': 2.145e-05, 'upstream_inference_completions_cost': 2.1e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_ab0a2ab924', 'id': 'gen-1786359846-X4kuMA15cfG1MYHkX1ET', 'finish_reason': 'tool_calls', 'logpr

## B3. Hard

**B3.1 — Long-term memory with `ToolRuntime`.** Build two tools: `save_dietary_preference`
(takes `customer_id`, `preference`, and `runtime: ToolRuntime`, saves to `runtime.store`) and
`recall_dietary_preference` (takes `customer_id` and `runtime: ToolRuntime`, reads it back).
Build an agent with these two tools and a real `store=` attached. Prove the memory survives
across two **separate** `.invoke()` calls — save a preference in the first call, and recall it
correctly in a second, independent call.


In [21]:
@tool
def save_dietary_preference(customer_id: str, preference: str, runtime: ToolRuntime) -> str:
    """
    Save a customer's dietary preference in the persistent store.
    """
    namespace = ("dietary_preferences",)
    runtime.store.put(namespace, customer_id, preference)
    return f"Saved preference for {customer_id}: {preference}"

@tool
def recall_dietary_preference(customer_id: str, runtime: ToolRuntime) -> str:
    """
    Recall a previously saved dietary preference for a customer from the store.
    """
    namespace = ("dietary_preferences",)
    stored = runtime.store.get(namespace, customer_id)
    if stored is None:
        return f"No preference found for {customer_id}."
    return f"Preference for {customer_id}: {stored}"

store = InMemoryStore()

agent = create_agent(
    model,
    tools=[save_dietary_preference, recall_dietary_preference],
    system_prompt="You are GreenPlate, a restaurant assistant. Use the available tools when asked to save or recall dietary preferences.",
    store=store,
)

# First call: save a preference
result1 = agent.invoke({
    "messages": [HumanMessage(content="Use the save_dietary_preference tool to save that customer C001 prefers vegan food.")]
})
print("First call:")
print(result1["messages"][-1].content)

# Second call: recall the same preference
result2 = agent.invoke({
    "messages": [HumanMessage(content="Use the recall_dietary_preference tool to recall the dietary preference for customer C001.")]
})
print("\nSecond call:")
print(result2["messages"][-1].content)

First call:
The dietary preference for customer C001 has been successfully saved as vegan.

Second call:
The dietary preference for customer C001 is vegan.


**B3.2 — Dynamic tool gating.** GreenPlate has a `book_private_dining_room` tool that should
only be available to customers with a "premium" membership tier. Using `wrap_model_call`, write
a middleware function that removes this tool from the model's visible toolset unless
`request.state.get("is_premium_member")` is `True`. Prove it works by running the SAME query
twice — once without the flag, once with it — and show the tool is genuinely unavailable in the
first case (not just "declined").


In [24]:
# Your solution for B3.2
from langchain.agents.middleware import wrap_model_call

@tool
def book_private_dining_room(customer_name: str, time_slot: str) -> str:
    """
    Book a private dining room for a customer.
    This tool should only be visible to premium members.
    """
    return f"Private dining room booked for {customer_name} at {time_slot}."


@wrap_model_call
def gate_private_dining_room(request, call_next):
    # Read premium flag from request state (or configurable config)
    state = getattr(request, "state", {}) or {}
    config = getattr(request, "config", {}) or {}
    configurable = config.get("configurable", {}) if isinstance(config, dict) else {}

    if "is_premium_member" not in state:
        state["is_premium_member"] = configurable.get("is_premium_member", False)

    # Hide the private-room tool unless the user is premium
    tools = getattr(request, "tools", None) or []
    if not state.get("is_premium_member", False):
        request.tools = [
            t for t in tools if getattr(t, "name", None) != "book_private_dining_room"
        ]

    print("Visible tools to model:", [getattr(t, "name", None) for t in request.tools])

    return call_next(request)

agent = create_agent(
    model,
    tools=[check_table_availability, book_private_dining_room],
    system_prompt="You are GreenPlate, a restaurant assistant.",
    middleware=[gate_private_dining_room],
)

query = "Please book the private dining room for Maya at 8:00 PM."

for premium in [False, True]:
    print(f"\n--- premium={premium} ---")
    result = agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config={"configurable": {"is_premium_member": premium}}
    )
    print(result["messages"][-1].content)


--- premium=False ---
Visible tools to model: ['check_table_availability']


C:\Users\visha\AppData\Local\Temp\ipykernel_29304\1912388285.py:26: DeprecationWarning: Direct attribute assignment to ModelRequest.tools is deprecated. Use request.override(tools=...) instead to create a new request with the modified attribute.
  request.tools = [


Could you please provide the party size for the reservation?

--- premium=True ---
Visible tools to model: ['check_table_availability']


C:\Users\visha\AppData\Local\Temp\ipykernel_29304\1912388285.py:26: DeprecationWarning: Direct attribute assignment to ModelRequest.tools is deprecated. Use request.override(tools=...) instead to create a new request with the modified attribute.
  request.tools = [


Could you please provide me with the party size for Maya's reservation?


**B3.3 — Combine structured output and tools in one agent.** Build a `create_agent` that has
BOTH a `check_table_availability`-style tool AND a `response_format=ReservationConfirmation`
schema (define this schema yourself — it should capture at minimum: customer_name, time_slot,
confirmed: bool). Send a request that requires the agent to actually call the tool to check
availability *before* it can correctly fill in `confirmed`. Print both `result["messages"]` and
`result["structured_response"]`, and explain in a comment why this required the agent-level
`response_format`, not the raw model-level `with_structured_output()`.


In [ ]:
# Your solution for B3.3


---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for GreenPlate that combines **at least five** of the
following in one working system (your choice which five, but justify your choices in a markdown
cell before your code):

- A structured schema for orders or reservations (with at least one real constraint, like a
  `Literal` or a numeric range)
- At least two custom tools
- Long-term memory via `ToolRuntime.store` (e.g. remembering a customer's dietary preferences or
  favorite dish across sessions)
- Short-term memory via a checkpointer and `thread_id` (e.g. remembering the customer's name
  within one conversation)
- Dynamic tool gating based on some condition (membership tier, time of day, order size — your
  choice)
- A `context_schema` carrying some per-run data a tool reads (e.g. `restaurant_location`)

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls that demonstrate the system actually working — not just that
   it builds without error.
4. A short markdown reflection (3-5 sentences) on ONE trade-off or limitation of your design —
   what would break, or what would you need to add, if this went to real production use.

This is intentionally open-ended. There is no single correct architecture — the goal is
demonstrating you can combine these pieces into something coherent, not matching a hidden answer
key.


*Design explanation goes here (before your code):*


In [ ]:
# Your capstone solution


*Your reflection on trade-offs/limitations goes here:*
